# 自动化交易部署：分步总览（Visual Step-by-Step Overview）

本节把此前的账户准备、基础设施、在线交易、日志和监控串成一个完整流程：准备天勤账户 → 准备云主机 → 配置 Python 环境 → 上传文件 → 启动交易程序 → 在本地监控。

本文使用天勤 TqSdk 和螺纹钢示例，衔接现有的 `run_trading.py`,`monitor_strategy.py` 与 `trading_common.py`，不重复编写函数。当前交易脚本连接的是天勤本地模拟账户 `TqSim`。

本地模型和文件检查已执行，结果保存在下方；天勤连接单元和终端命令待手动执行，云主机部署、实时交易和远程监控均未启动。

## 1. 准备天勤账户

第一步是准备天勤行情认证账户，并将认证信息保存到 `tqsdk.cfg`。

### 1.1. 连接天勤并检查模拟账户

在准备好 `tqsdk.cfg` 后，手动运行下面的单元。它会读取认证配置，连接天勤服务，并获取本地 `TqSim` 模拟账户信息。此单元不订阅行情、不下单；结束时由 `with` 自动关闭连接。


In [4]:
from trading_common import make_api

with make_api() as api:  # 读取配置，完成天勤认证并创建 TqSim 模拟账户
    account = api.get_account()
    print('天勤 API 已连接，当前使用 TqSim 模拟账户。')
    print(f'账户权益：{account.balance:,.2f} 元')


天勤 API 已连接，当前使用 TqSim 模拟账户。
账户权益：100,000.00 元


本单元尚未执行，因此不展示连接成功或账户余额的结果。手动运行后，应以实际输出或错误信息判断连接状态。这里检查的是行情认证与本地模拟账户，不能据此判断实际委托能否成交。后续启动 `run_trading.py` 时，脚本会自行调用 `make_api()` 建立新的连接。

### 1.2. 核对预测模型和仓位配置

原书在这一步还演示了按 Kelly 准则设置账户杠杆，改用国内期货后，不能直接照搬账户界面上的杠杆设置。

前文虽然估算了 Kelly 杠杆，当前脚本仍使用固定一手，没有自动加入 Kelly 仓位管理。下面核对模型的标的、周期和默认目标手数：

In [5]:
from pathlib import Path
import pickle
from trading_common import MLTrader

with Path('algorithm.pkl').open('rb') as f:
    algorithm = pickle.load(f)
trader = MLTrader(algorithm)

print('信号标的：', trader.instrument)
print('K 线周期：', trader.bar_seconds, '秒')
print('模型输入特征数：', len(trader.cols))
print('多头目标：', trader.volume, '手')
print('空头目标：', -trader.volume, '手')


信号标的： KQ.m@SHFE.rb
K 线周期： 600 秒
模型输入特征数： 36
多头目标： 1 手
空头目标： -1 手


输出说明，当前模型使用螺纹钢主力连续的 600 秒 K 线，接收 36 个输入特征。默认多头、空头目标分别为 `+1` 和 `−1` 手。这里的一手是合约数量，不表示一倍杠杆。

加载模型并创建 `MLTrader` 只准备了预测对象，不会连接账户。上一单元用于单独检查天勤连接；实际行情订阅、主力合约映射和交易委托由第 5 节的交易脚本完成。

## 2. 准备腾讯云轻量应用服务器

本例改用腾讯云轻量应用服务器，体验将交易程序部署到远程主机的过程。由于本次只用于学习，优先领取免费试用，不需要购买长期套餐。进入 [腾讯云免费体验馆](https://cloud.tencent.com/act/pro/free)，完成个人实名认证后，查看自己是否符合轻量应用服务器的试用资格。免费试用可作为本章部署练习的起点。实际可领取的规格、地域与库存，以登录后的页面为准；如果没有试用资格，不必为了完成本节购买付费套餐。

### 2.1. 创建试用实例

在试用页面选择轻量应用服务器，按页面提示完成配置：

1. 地域：选择试用套餐允许的中国内地地域。实例建立后，再验证它与天勤服务的连接情况。
2. 镜像：选择 Ubuntu 系统镜像，后续自行安装 Python 和项目依赖。
3. 实例名称：可以填写 `trading-learning`，便于在控制台识别。
4. 费用与续费：确认订单应付金额为 **0 元**，不勾选自动续费，不额外加购云硬盘等资源，再提交领取。

等待实例状态变为“运行中”后，进入实例详情页，记录它的公网 IP。这是后续从本地登录服务器、上传文件和建立 SSH 转发时要用到的地址。

### 2.2. 登录服务器

实例创建后，可在腾讯云站内信中找到本次实例的登录信息，包括用户名、初始登录密码和公网 IP。

可以通过腾讯云控制台的网页终端登录，也可以直接在自己的电脑上用 SSH 登录。使用本地 SSH 时，确认轻量服务器防火墙允许自己的连接访问 SSH 端口（默认 22）。

如何在自己的电脑上用 SSH 登录？在**本地**电脑终端执行以下命令，将用户名和公网 IP 替换为站内信中提供的信息：

```bash
ssh 用户名@服务器公网IP
```

例如，用户名为 `ubuntu` 时，命令是 `ssh ubuntu@服务器公网IP`。首次连接会询问是否信任该主机，核对后输入 `yes`；随后按提示输入站内信中的初始登录密码。登录成功后，终端会进入云主机的命令行。此时输入的命令在腾讯云服务器上执行，而不是在本地电脑上执行。下面在**云主机**终端创建项目目录：

```bash
mkdir -p ~/trading
cd ~/trading
```

后续将程序、模型和配置文件上传到这个目录， 并配置相应的环境。

### 2.3. 学习结束后释放资源

学完部署流程后，将需要保留的代码和日志下载到本地，再到控制台删除或销毁试用实例，并检查是否还有保留的独立云硬盘、快照等资源。无需为了保留本次学习环境而续费。

更多基础设施的可靠性、性能和安全性要求见 [Infrastructure_and_Deployment.ipynb](Infrastructure_and_Deployment.ipynb)。


## 3. 配置 Python 环境

本节在腾讯云 Ubuntu 服务器上安装 Python 和项目依赖。以下命令全部输入到登录后的本地服务器终端中。

现有模型使用 Python 3.11，scikit-learn 1.9.0 训练。本节使用带 Python 3.11 的 Miniconda 安装包提供解释器，再创建项目专用的 `.venv` 环境，避免依赖 Ubuntu 自带的 Python 版本，也不改动系统 Python。这里不使用额外的 conda 环境或 `conda activate`；后续统一激活 `.venv`。

### 3.1. 检查系统并安装下载工具

```bash
cat /etc/os-release
uname -m
sudo apt update
sudo apt install -y curl ca-certificates
```

这套命令适用于 **Ubuntu，x86_64 架构** 实例。`uname -m` 应输出 `x86_64`；如果输出 `aarch64`，应改用对应的 ARM 安装包，不能继续使用下面的 x86_64 安装包。

`sudo` 若提示输入密码，使用服务器登录密码；

### 3.2. 安装 Python 3.11，创建项目环境 

下载指定的 Python 3.11 安装包，并安装到自己的用户目录：

```bash
cd ~/trading
curl -fL --retry 3 --connect-timeout 15 \
  -o miniconda311.sh \
  https://mirrors.cqupt.edu.cn/anaconda/miniconda/Miniconda3-py311_25.7.0-2-Linux-x86_64.sh

bash miniconda311.sh -b -p "$HOME/miniconda311"
"$HOME/miniconda311/bin/python" --version

"$HOME/miniconda311/bin/python" -m venv .venv
source .venv/bin/activate
python --version
which python
```

`--version` 应显示 `Python 3.11.x`；`which python` 应指向类似 `/home/ubuntu/trading/.venv/bin/python` 的路径。安装包的补丁版本不一定与本地完全相同，这里固定的是 Python 3.11 系列，第三方核心软件版本在下一步锁定。

`-b` 表示按安装器的批处理方式安装，不需要执行 `conda init`。这一步只需完成一次；如果已经成功安装并创建环境，重新登录后直接按第 3.5 节激活即可，不要再次安装到同一个目录。

若国内镜像下载失败，可将下载命令中的地址替换为 [Anaconda 官方同一安装包](https://repo.anaconda.com/miniconda/Miniconda3-py311_25.7.0-2-Linux-x86_64.sh)。安装器约 150 MiB，下载和安装时等待命令执行完成，再继续下一行。

### 3.3. 写入依赖清单并安装

项目目录现已准备好 `requirements.txt`，内容来自本地已验证的运行环境。如果尚未上传这个文件，可以直接在**云主机终端**完整复制下面一块命令，现场生成同样的文件：

```bash
cd ~/trading
cat > requirements.txt <<'EOF'
# Runtime dependencies matching the locally validated quant-ml environment.
# Python 3.11
numpy==2.4.6
pandas==3.0.3
scipy==1.17.1
scikit-learn==1.9.0
tqsdk==3.10.2
pyzmq==27.1.0
EOF
```

继续在已激活的 `.venv` 中执行安装。这里使用 [腾讯云官方文档介绍的 PyPI 镜像](https://cloud.tencent.com/document/product/213/8623)：

```bash
source ~/trading/.venv/bin/activate
python -m pip install --no-cache-dir --timeout 60 \
  -i https://mirrors.cloud.tencent.com/pypi/simple \
  -r ~/trading/requirements.txt
python -m pip check
```

正常情况下，安装完成后回到命令提示符，`pip check` 显示 `No broken requirements found.`。核心软件版本已固定，其余依赖由 pip 自动解析安装，不必逐个手工安装。

如果镜像访问失败，或镜像尚未同步某个指定版本，保持依赖版本不变，改用官方 PyPI 源重试：

```bash
python -m pip install --no-cache-dir --timeout 60 \
  -i https://pypi.org/simple \
  -r ~/trading/requirements.txt
python -m pip check
```

当前部署使用已有的两个终端脚本，不需要先安装 JupyterLab。以后需要在云端打开 notebook 时，再单独安装它。

### 3.4. 验证 Python 和软件包

依赖安装成功后，在**云主机终端**完整执行下面的检查：

```bash
python - <<'PY'
import sys
from importlib.metadata import version
import numpy
import pandas
import scipy
import sklearn
import tqsdk
import zmq

print('Python 解释器：', sys.executable)
print('Python 版本：', sys.version.split()[0])
for package in ['numpy', 'pandas', 'scipy', 'scikit-learn', 'tqsdk', 'pyzmq']:
    print(f'{package}: {version(package)}')
print('核心依赖导入成功。')
PY
```

检查时确认解释器位于 `~/trading/.venv/` 中，列出的软件版本与 `requirements.txt` 一致。TqSdk 导入时可能显示免责声明，这是软件自身的输出；导入 TqSdk 不会自动连接账户。

这里尚不需要 `algorithm.pkl` 或 `tqsdk.cfg`。它们在第 4 节上传，第 5 节运行交易脚本时才会加载模型并连接天勤。这样可以先把环境准备好，再核对项目文件，不必在环境安装阶段启动交易程序。

### 3.5. 下次登录时如何继续

退出 SSH 后再次登录，Python 和依赖仍保留在服务器上，只需要重新进入目录并激活环境：

```bash
cd ~/trading
source .venv/bin/activate
```

看到提示符出现 `(.venv)` 后，就可以继续后面的上传与运行步骤。若想退出项目环境，执行 `deactivate` 即可。

本地已核对模型保存的 scikit-learn 版本和依赖清单，检查了安装包下载地址，并通过 PyPI 的只读安装预检确认六个指定版本都有 Python 3.11/Linux 可用的安装包。上面的终端命令也通过了 Bash 语法检查；尚未登录你的腾讯云实例执行安装，云端结果应以逐步操作时的真实输出为准。

## 4. 上传代码、模型与账户配置

第四步是将程序及所需文件上传到云主机，对应原书图 10-12。这里复用此前已整理的文件：

| 文件 | 用途 |
|---|---|
| `run_trading.py` | 运行预测、模拟交易、日志记录和消息发布 |
| `monitor_strategy.py` | 独立接收和打印交易消息；本地电脑也需保留一份 |
| `trading_common.py` | 共用的特征构造、预测及天勤模拟连接 |
| `algorithm.pkl` | 已训练模型、标准化参数及训练配置 |
| `tqsdk.cfg` | 天勤行情认证配置 |
| `requirements.txt` | 第 3 节已生成并安装的软件依赖清单 |

上传时保留这些文件之间的相对位置。`run_trading.py` 会切换到脚本所在目录，再读取模型和配置文件。模型文件不包含特征构造代码，所以仍需上传公共模块。

下面核对本地现有核心文件是否齐备，只查看文件是否存在，不读取或显示账户密码。`requirements.txt` 已按本地运行环境生成；如果已在第 3 节现场创建，则无需重复上传。

In [2]:
required_files = [
    'run_trading.py',
    'monitor_strategy.py',
    'trading_common.py',
    'algorithm.pkl',
    'tqsdk.cfg',
    'requirements.txt',
]
for name in required_files:
    status = '存在' if Path(name).is_file() else '缺失'
    print(f'{name:<24} {status}')


run_trading.py           存在
monitor_strategy.py      存在
trading_common.py        存在
algorithm.pkl            存在
tqsdk.cfg                存在
requirements.txt         存在


**运行结果分析**

本地核心文件的存在状态如上。这个检查只说明文件是否位于当前工作目录，不能证明它们已经上传到云主机，也没有验证认证信息、网络连接或实际成交。

可通过云服务商的文件上传界面，或使用 SCP/SFTP 将文件放入 `~/trading`。账户配置含认证信息，应通过安全连接传输，不应贴入公共日志或提交到公开代码仓库。

## 5. 启动自动化交易程序

第五步是在云主机终端运行交易脚本，对应原书图 10-13；图 10-14 展示了脚本产生的一笔交易。本项目直接使用已有的 `run_trading.py`。

以下为**手动运行示例，尚未执行**：

```bash
cd ~/trading
source .venv/bin/activate
python run_trading.py --stop 500 --timeout 60
```

程序加载模型、连接天勤模拟账户，使用主力连续行情生成信号，再通过其当前对应的实际合约设置目标持仓。日志保存在脚本目录的 `automated_strategy.log`，同一份消息也发布到默认地址 `tcp://127.0.0.1:5555`。按当前脚本设计，每次启动会重新创建日志文件。

`--stop 500` 表示最多累计 500 次报价更新，`--timeout 60` 表示连续 60 秒没有报价更新就退出。每结束一根 10 分钟 K 线才生成预测；如果本次运行未等到新 K 线，可能没有预测或委托。退出时脚本尝试平仓，最多再等待 30 秒，并记录实际状态。

**运行结果说明**

本 notebook 没有启动该交易程序，因此这里没有新的成交结果。真实运行时，应分别查看模型目标和实际成交：`GOING LONG/SHORT (TARGET)` 表示目标方向变化；当前脚本用 `EXECUTED` 标记收到的实际成交。只看到启动或目标日志，不能认定已经成交。

## 6. 在本地实时监控

最后一步是在本地电脑启动监控程序，对应原书图 10-15。交易程序继续运行在服务器上，本地只接收它发布的消息；关闭监控端不会自动停止服务器上的交易程序。

当前交易脚本默认只监听服务器的 `127.0.0.1:5555`。为复用该配置，可以通过 SSH 转发把服务器的这个端口接到本地。下面的命令在**本地终端 A** 执行，将用户名与地址替换为自己的信息，保持此终端运行：

```bash
ssh -N -L 5555:127.0.0.1:5555 用户名@云主机地址
```

然后在**本地终端 B** 中进入保存监控脚本的目录，激活相应 Python 环境，再执行：

```bash
python monitor_strategy.py --address tcp://127.0.0.1:5555 --timeout 900
```

此处 `900` 表示连续 15 分钟收不到监控消息才退出，便于等待 10 分钟 K 线产生预测。它等待的是策略发布的日志消息，与交易端等待新报价的 `--timeout` 含义不同。

建议先建立 SSH 转发、启动监控端，再启动第 5 步的交易程序。ZeroMQ 的 PUB/SUB 不补发订阅连接建立之前的消息，最早的消息仍可能错过；完整记录应以服务器上的日志文件为准。

原书脚注指出，示例 socket 通信本身不加密，会通过网络传送明文。这里使用 SSH 转发保护跨网络的传输，并继续使用现有脚本的本机监听地址。若交易端和监控端都在同一台电脑上，可省略 SSH 转发，分别在两个终端运行即可。

**运行结果说明**

本次没有连接云主机或运行远程监控。实际启动后，监控窗口显示交易程序发出的时间、输入、信号、目标与成交信息；这些消息来自程序运行，不是监控端重新预测或下单。

## 7. 本节要点

本节将此前内容串成部署顺序：准备天勤账户和模拟仓位配置，选择运行环境，配置 Python，上传代码、模型与认证文件，再分别启动交易端和监控端。

交易端负责处理行情、预测、执行模拟交易并发布日志；本地监控端负责接收和展示消息。账户准备、文件上传或程序启动成功，都需要与实际行情处理和成交结果区分。